# 09 · Tabelas e figuras

Tabelas e figuras do relatório, feitas só com o que as etapas 01 a 08 salvaram. Cada tabela sai em `.csv` e `.tex`.

In [ ]:
import sys, json
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config

ETAPA = "09_tabelas_figuras"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "grid.alpha": .3, "savefig.dpi": 150, "savefig.bbox": "tight"})

def ler(etapa, arquivo, **kw):
    return pd.read_csv(config.ler_resultado(etapa, arquivo), **kw)

tabela = ler("05_tabela", "tabela_mensal.csv", index_col="mes")
tabela.index = pd.PeriodIndex(tabela.index, freq="M")
COLUNAS = json.loads(config.ler_resultado("06_features", "config.json").read_text())["colunas_finais"]
NOMES = {"passeio": "Passeio aleatório", "ultimo_dia": "Último dia", "escolhido": "Ridge (etapa 06)",
         "ridge": "Ridge sobre o desvio", "rf": "Random Forest", "lstm": "LSTM", "bilstm": "BiLSTM"}

def salvar_tabela(df, nome, legenda):
    df.to_csv(PASTA / f"{nome}.csv", index=False)
    (PASTA / f"{nome}.tex").write_text(df.to_latex(index=False, caption=legenda, label=f"tab:{nome}",
                                                   float_format="%.3f", escape=True))
    return df

def salvar_figura(nome):
    plt.savefig(PASTA / f"{nome}.png"); plt.show()

## Tabela 1 · Features do modelo final

In [ ]:
DESCRICAO = {
    "dlog_preco": ("Preço", "FMI", "Variação do preço no mês (log)"),
    "dlog_preco_lag1": ("Preço", "FMI", "Variação do preço no mês anterior"),
    "tendencia_3m": ("Preço", "FMI", "Variação do preço em 3 meses"),
    "mes_sin": ("Sazonalidade", "Calendário", "Seno do mês do ano"),
    "mes_cos": ("Sazonalidade", "Calendário", "Cosseno do mês do ano"),
    "fim_vs_media": ("Fim do mês", "Yahoo (ZS=F)", "Último dia de Chicago menos a média do mês (log)"),
    "ultima_semana": ("Fim do mês", "Yahoo (ZS=F)", "Variação nos últimos 5 pregões do mês"),
    "oni": ("ENSO", "NOAA CPC", "Índice ONI do mês"),
    "oni_lag2": ("ENSO", "NOAA CPC", "ONI de 2 meses antes"),
    "oni_lag8": ("ENSO", "NOAA CPC", "ONI de 8 meses antes"),
    "oni_var3": ("ENSO", "NOAA CPC", "Variação do ONI em 3 meses"),
    "fundos_liquido": ("Fundos", "CFTC COT", "Posição líquida dos fundos / contratos em aberto"),
    "fundos_var": ("Fundos", "CFTC COT", "Variação da posição líquida no mês"),
}
t1 = pd.DataFrame([(c, *DESCRICAO[c]) for c in COLUNAS], columns=["Feature", "Grupo", "Fonte", "Descrição"])
salvar_tabela(t1, "tab1_features", "Features usadas no modelo final.")

## Tabela 2 e Figura 3 · Correlação com o alvo

In [ ]:
candidatas = [c for c in tabela.columns if c not in ("preco_usd_t", "alvo")]
antes, depois = tabela.loc[:"2014-12"], tabela.loc[config.TESTE_INICIO:config.TESTE_FIM]
t2 = pd.DataFrame({"Feature": candidatas,
                   "Até 2014": [antes[c].corr(antes["alvo"]) for c in candidatas],
                   "2015 em diante": [depois[c].corr(depois["alvo"]) for c in candidatas]})
t2["No modelo"] = np.where(t2["Feature"].isin(COLUNAS), "sim", "")
t2 = t2.reindex(t2["Até 2014"].abs().sort_values(ascending=False).index).reset_index(drop=True)
salvar_tabela(t2, "tab2_correlacao", "Correlação de cada feature com a variação do preço no mês seguinte.")

top = t2.head(15).iloc[::-1]
plt.figure(figsize=(7, 5))
plt.barh(top["Feature"], top["Até 2014"], color=np.where(top["No modelo"] == "sim", "tab:blue", "lightgray"))
plt.axvline(0, c="k", lw=.8); plt.xlabel("correlação com o alvo (até 2014)")
plt.title("15 maiores correlações (azul: no modelo final)")
salvar_figura("fig3_correlacao")

## Figura 1 · Preço da soja e períodos

In [ ]:
p = tabela["preco_usd_t"]
plt.plot(p.index.to_timestamp(), p, "k", lw=1)
for (ini, fim), cor, nome in [((config.VALIDACAO_INICIO, config.VALIDACAO_FIM), "tab:orange", "validação"),
                              ((config.TESTE_INICIO, config.TESTE_FIM), "tab:blue", "teste")]:
    plt.axvspan(pd.Period(ini, "M").to_timestamp(), pd.Period(fim, "M").to_timestamp(), color=cor, alpha=.12, label=nome)
plt.ylabel("US$/t"); plt.legend(loc="upper left"); plt.title("Preço mensal da soja (FMI)")
salvar_figura("fig1_preco")

## Figura 2 · Efeito da média mensal

In [ ]:
d = tabela[["fim_vs_media", "alvo"]].dropna()
r = d.corr().iloc[0, 1]
plt.figure(figsize=(5, 4.5))
plt.scatter(d["fim_vs_media"], d["alvo"], s=10, alpha=.5)
lim = np.array([d.min().min(), d.max().max()])
plt.plot(lim, lim, "k--", lw=.8, label="alvo = fim_vs_media")
plt.xlabel("fim_vs_media (último dia − média do mês)"); plt.ylabel("alvo (variação no mês seguinte)")
plt.legend(); plt.title(f"Fim do mês contra o alvo (r = {r:.2f})")
salvar_figura("fig2_media_mensal")

## Tabela 3 e Figura 4 · Grupos de features (etapa 06)

In [ ]:
r06 = ler("06_features", "resumo.csv")
g = r06[(r06["periodo"] == "validacao") & ~r06["modelo"].isin(["passeio", "ultimo_dia"])]
t3 = pd.DataFrame({"Grupo": g["modelo"], "RMSE rel. passeio": g["rmse_rel_passeio"], "RMSE rel. último dia": g["rmse_rel_ultimo_dia"]}
                  ).sort_values("RMSE rel. passeio").reset_index(drop=True)
salvar_tabela(t3, "tab3_grupos", "Ridge com a base mais cada grupo de features, na validação (2010 a 2014).")

plt.figure(figsize=(7, 4))
plt.barh(t3["Grupo"][::-1], t3["RMSE rel. passeio"][::-1], color="tab:blue")
plt.axvline(1, c="k", lw=.8)
plt.axvline(r06.query("modelo == 'ultimo_dia' and periodo == 'validacao'")["rmse_rel_passeio"].item(), c="tab:red", ls="--", label="último dia")
plt.xlim(.6, 1.02); plt.xlabel("RMSE relativo ao passeio (validação)"); plt.legend(); plt.title("Cada grupo somado à base")
salvar_figura("fig4_grupos")

## Tabela 4 e Figura 5 · Modelos no teste

In [ ]:
r07 = ler("07_modelos", "resumo.csv")
t4 = pd.concat([r06[(r06["periodo"] == "teste") & r06["modelo"].isin(["passeio", "ultimo_dia", "escolhido"])],
                r07[r07["periodo"] == "teste"]], ignore_index=True)
t4.loc[t4["modelo"] == "passeio", "rmse_rel_passeio"] = 1.0
t4.loc[t4["modelo"] == "ultimo_dia", "rmse_rel_ultimo_dia"] = 1.0
t4.loc[t4["modelo"] == "passeio", "direcao"] = np.nan  # o passeio prevê variação zero, não tem direção
t4 = pd.DataFrame({"Modelo": t4["modelo"].map(NOMES), "RMSE": t4["rmse"], "MAE": t4["mae"], "Direção": t4["direcao"],
                   "rel. passeio": t4["rmse_rel_passeio"], "p (passeio)": t4["p_passeio"],
                   "rel. último dia": t4["rmse_rel_ultimo_dia"], "p (último dia)": t4["p_ultimo_dia"]})
salvar_tabela(t4, "tab4_modelos", "Resultados no teste (jan/2015 a jun/2026, 138 meses). p: teste de Diebold-Mariano.")

m = t4.iloc[2:]
x = np.arange(len(m))
plt.figure(figsize=(9, 3.8))
plt.bar(x - .2, m["rel. passeio"], .4, label="rel. passeio")
plt.bar(x + .2, m["rel. último dia"], .4, label="rel. último dia")
plt.axhline(1, c="k", lw=.8); plt.xticks(x, m["Modelo"].str.replace(" sobre ", "\nsobre ").str.replace(" (etapa", "\n(etapa")); plt.ylim(.6, 1.15)
plt.ylabel("RMSE relativo"); plt.legend(); plt.title("Erro no teste em relação às duas referências")
salvar_figura("fig5_modelos")

## Figura 6 e 7 · Previsões no teste

In [ ]:
p07 = ler("07_modelos", "previsoes.csv")
p07 = p07[p07["periodo"] == "teste"].assign(mes=lambda d: pd.PeriodIndex(d["mes"], freq="M"))
ridge = p07[p07["modelo"] == "ridge"].set_index("mes")
meses = ridge.index
base = tabela.loc[meses, "preco_usd_t"]
quando = (meses + 1).to_timestamp()
plt.plot(quando, base * np.exp(ridge["alvo"]), "k", lw=1, label="real")
plt.plot(quando, base * np.exp(tabela.loc[meses, "fim_vs_media"]), c="tab:gray", lw=.8, ls="--", label="último dia")
plt.plot(quando, base * np.exp(ridge["previsao"]), c="tab:red", lw=1, label="Ridge")
plt.ylabel("US$/t"); plt.legend(loc="upper left"); plt.title("Preço do mês seguinte no teste")
salvar_figura("fig6_previsoes")

real, ultimo = ridge["alvo"], tabela.loc[meses, "fim_vs_media"]
for modelo in ["ridge", "rf", "lstm", "bilstm"]:
    prev = p07[p07["modelo"] == modelo].set_index("mes")["previsao"].reindex(meses)
    plt.plot(quando, ((real - ultimo) ** 2 - (real - prev) ** 2).cumsum(), label=NOMES[modelo])
plt.axhline(0, c="k", lw=.8); plt.ylabel("erro² acumulado do último dia − do modelo")
plt.legend(); plt.title("Vantagem acumulada sobre o último dia (acima de 0 = modelo melhor)")
salvar_figura("fig7_vantagem")

## Tabela 5 · Hiperparâmetros

In [ ]:
hp = json.loads(config.ler_resultado("07_modelos", "hiperparametros.json").read_text())
c08 = json.loads(config.ler_resultado("08_modelo_final", "config.json").read_text())
linhas = [("Ridge", "alpha (modelo final)", c08["alpha"])]
linhas += [(NOMES[m], k, v) for m, params in hp.items() for k, v in params.items()]
t5 = pd.DataFrame(linhas, columns=["Modelo", "Hiperparâmetro", "Valor"])
t5["Valor"] = t5["Valor"].map(lambda v: f"{v:.4g}" if isinstance(v, float) else str(v))
salvar_tabela(t5, "tab5_hiperparametros", "Hiperparâmetros escolhidos na validação (Optuna, 30 tentativas) e alpha do Ridge final.")

## Tabela 6, Figuras 8 e 9 · Modelo final

In [ ]:
prev = json.loads(config.ler_resultado("08_modelo_final", "previsao.json").read_text())
t6 = pd.DataFrame([("Mês previsto", prev["mes_previsto"]),
                   (f"Preço em {prev['mes_base']} (US$/t)", f"{prev['preco_atual']:.1f}"),
                   ("Previsão do Ridge (US$/t)", f"{prev['preco_previsto']:.1f}"),
                   ("Previsão do último dia (US$/t)", f"{prev['preco_ultimo_dia']:.1f}"),
                   *[(f"Faixa de {n} (US$/t)", f"{a:.0f} a {b:.0f}") for n, (a, b) in prev["faixas"].items()]],
                  columns=["Item", "Valor"])
salvar_tabela(t6, "tab6_previsao", "Previsão do modelo final para o próximo mês.")

mes_base = pd.Period(prev["mes_base"], "M")
hist = tabela.loc[mes_base - 35:mes_base, "preco_usd_t"]
x0, x1 = mes_base.to_timestamp(), (mes_base + 1).to_timestamp()
plt.plot(hist.index.to_timestamp(), hist, "k", label="preço FMI")
for n, cor in [("95%", "mistyrose"), ("80%", "lightsalmon")]:
    a, b = prev["faixas"][n]
    plt.fill_between([x0, x1], [prev["preco_atual"], a], [prev["preco_atual"], b], color=cor, label=n)
plt.plot([x0, x1], [prev["preco_atual"], prev["preco_previsto"]], "o-", c="tab:red", label="previsão")
plt.ylabel("US$/t"); plt.legend(loc="upper left"); plt.title(f"Previsão para {prev['mes_previsto']}")
salvar_figura("fig8_previsao")

coef = ler("08_modelo_final", "coeficientes.csv").set_index("feature")["coeficiente"].sort_values()
plt.figure(figsize=(7, 4))
plt.barh(coef.index, coef, color=np.where(coef > 0, "tab:green", "tab:red"))
plt.axvline(0, c="k", lw=.8); plt.xlabel("coeficiente (features padronizadas)"); plt.title("Pesos do Ridge final")
salvar_figura("fig9_coeficientes")

## Salvando

In [ ]:
config.salvar_config(PASTA, fontes=["01_preco", "05_tabela", "06_features", "07_modelos", "08_modelo_final"])
print(sorted(p.name for p in PASTA.iterdir()))